# 설치
  - `pip install langchain langchain-qdrant langchain-openai python-dotenv`

In [1]:
!uv pip install langchain-qdrant

Resolved 40 packages in 1.60s
Prepared 1 package in 34ms
Installed 1 package in 7ms
 + langchain-qdrant==1.1.0


# Vector Store 생성
- QdrantClient로 Database에 연결
    - Collection 생성(없으면)
- VectorSore 생성
    - client와 collection name, embedding 모델을 전달해서 생성

In [17]:
str(uuid4())

'8fa10359-efeb-469b-a2d4-69035b17d4fa'

In [1]:
from uuid import uuid4
from langchain_core.documents import Document

def get_documents():
    data = [
        {
            "name": "The Time Machine",
            "description": "시간 여행자는 미래로 여행을 떠나 인류의 진화와 몰락을 목격하게 된다. 그는 멀리 떨어진 미래 사회에서 인간 문명이 퇴화한 모습을 보고 충격을 받는다. 이 이야기는 인간 문명과 과학기술의 진보가 가져올 결과를 철학적으로 탐구한다.",
            "author": "H.G. Wells",
            "year": 1895
        },
        {
            "name": "Ender's Game",
            "description": "소년 엔더는 외계 종족 '버거'와의 전쟁을 대비해 혹독한 군사 훈련을 받는다. 그는 전략적 천재성을 발휘하여 점차 전쟁의 중심에 서게 된다. 이야기는 어린아이를 도구로 삼는 전쟁의 비인간성과 심리적 고통을 그린다.",
            "author": "Orson Scott Card",
            "year": 1985
        },
        {
            "name": "Brave New World",
            "description": "유전자 조작과 세뇌로 계급이 철저히 나뉘어진 미래 사회가 배경이다. 인간은 자유 없이 쾌락과 안정을 주입받으며 살고, 비순응자는 배제된다. 디스토피아 속에서 인간 본성과 자유의 의미를 묻는 소설이다.",
            "author": "Aldous Huxley",
            "year": 1932
        },
        {
            "name": "The Hitchhiker's Guide to the Galaxy",
            "description": "지구 멸망 직후 우주로 탈출한 주인공이 외계 친구와 함께 기상천외한 모험을 시작한다. 우주의 황당한 존재들과 철학적 농담들이 넘치는 코믹 SF다. 우주에 대한 진지한 질문을 유쾌하게 풀어낸 작품이다.",
            "author": "Douglas Adams",
            "year": 1979
        },
        {
            "name": "Dune",
            "description": "사막 행성 아라키스를 둘러싼 정치적 음모와 권력 투쟁이 벌어진다. 주인공 폴은 운명과 예언에 따라 거대한 반란을 이끌게 된다. 정치, 종교, 생태학이 얽힌 거대한 서사의 서막이다.",
            "author": "Frank Herbert",
            "year": 1965
        },
        {
            "name": "Dune Messiah",
            "description": "황제가 된 폴 아트레이디스는 종교적 숭배와 정치적 음모 속에서 고뇌한다. 혁명의 영웅이 권력의 상징이 되었을 때 발생하는 부작용이 드러난다. 영웅 서사의 이면과 메시아 신화를 비판적으로 해체한 작품이다.",
            "author": "Frank Herbert",
            "year": 1969
        },
        {
            "name": "Children of Dune",
            "description": "폴의 자녀들이 초인적인 능력을 지닌 존재로 성장하며 제국의 운명을 짊어진다. 인간 진화와 권력의 세습이라는 주제가 본격적으로 탐구된다. 듄 세계관에서 철학적·생물학적 논의가 가장 심화된 작품 중 하나다.",
            "author": "Frank Herbert",
            "year": 1976
        },
        {
            "name": "Foundation",
            "description": "수학자 하리 셀던은 미래를 예측하는 '심리역사학'을 개발해 인류의 몰락을 막기 위한 계획을 세운다. 그는 제국의 붕괴 속에서도 문명을 재건할 수 있는 재단을 설립한다. 과학과 지식의 힘이 문명을 구원할 수 있는지를 탐구한다.",
            "author": "Isaac Asimov",
            "year": 1951
        },
        {
            "name": "I, Robot",
            "description": "로봇공학 3원칙을 기반으로 한 여러 단편들이 엮인 작품이다. 로봇과 인간의 상호작용 속에서 윤리적 딜레마와 예측 불가능한 상황이 발생한다. 인공지능과 책임, 통제의 문제를 선구적으로 탐구한 고전 SF다.",
            "author": "Isaac Asimov",
            "year": 1950
        },
        {
            "name": "Cryptonomicon",
            "description": "제2차 세계대전의 암호 전쟁과 현대 정보 사회가 교차 서술된다. 암호학, 컴퓨터 과학, 금융이 서사의 핵심 요소다. 기술과 자본, 정보의 힘을 대서사로 풀어낸 작품이다.",
            "author": "Neal Stephenson",
            "year": 1999
        },
        {
            "name": "The Caves of Steel",
            "description": "인구 과밀의 지구와 우주 거주민 사이의 갈등이 살인 사건을 계기로 드러난다. 인간 형사와 로봇 파트너가 함께 수사를 진행한다. 인간 사회와 로봇 공존의 가능성을 사회과학적 관점에서 분석한 작품이다.",
            "author": "Isaac Asimov",
            "year": 1953
        },
        {
            "name": "Snow Crash",
            "description": "가상현실 '메타버스'와 현실이 뒤얽힌 미래에서, 주인공 해커는 '스노 크래시'라는 치명적인 바이러스를 추적한다. 정보와 언어, 권력이 하나로 연결된 세계가 그려진다. 포스트사이버펑크의 대표작으로 인터넷 이후의 세계를 예견한다.",
            "author": "Neal Stephenson",
            "year": 1992
        },
        {
            "name": "Neuromancer",
            "description": "해커 '케이스'는 거대 기업의 의뢰를 받아 사이버 공간에서 불가능한 해킹을 수행한다. 임무를 수행하며 인공지능과 인간의 경계가 모호해지는 음모에 휘말린다. 사이버펑크 장르의 원형으로 가상 현실과 인간 의식의 융합을 다룬다.",
            "author": "William Gibson",
            "year": 1984
        },
        {
            "name": "The War of the Worlds",
            "description": "화성인이 지구를 침략하면서 인류는 공포와 절망 속에 빠진다. 압도적인 기술력 앞에 속수무책으로 무너지는 인간의 모습이 그려진다. 외계 생명체와 문명의 충돌을 통해 인간 중심적 세계관에 경종을 울린다.",
            "author": "H.G. Wells",
            "year": 1898
        },
        {
            "name": "The Invisible Man",
            "description": "과학자 그리핀은 투명화 실험에 성공하지만, 그 대가로 점점 인간성을 잃어간다. 사회로부터 고립된 그는 자신의 능력을 폭력과 공포의 수단으로 사용한다. 과학적 성취가 윤리와 결합되지 않을 때 어떤 비극이 발생하는지를 보여주는 작품이다.",
            "author": "H.G. Wells",
            "year": 1897
        },
        {
            "name": "The Island of Doctor Moreau",
            "description": "외딴 섬에서 미친 과학자 모로 박사는 동물들을 인간처럼 개조하는 실험을 진행한다. 주인공은 이 섬에서 인간과 짐승의 경계가 무너진 광경을 목격한다. 생명 윤리와 과학의 오만함을 날카롭게 비판하는 소설이다.",
            "author": "H.G. Wells",
            "year": 1896
        },
        {
            "name": "The Hunger Games",
            "description": "디스토피아 사회에서 어린이들은 생존 게임에 강제로 참가해야 한다. 주인공 캣니스는 잔혹한 게임 속에서 인간성과 저항의 상징이 되어간다. 권력과 억압, 생존 본능의 이야기가 펼쳐진다.",
            "author": "Suzanne Collins",
            "year": 2008
        },
        {
            "name": "The Andromeda Strain",
            "description": "우주에서 온 미생물이 미국의 한 마을을 초토화시키자, 과학자들이 이를 분석하고 대응에 나선다. 시간과의 싸움 속에서 인류의 과학 기술과 오만함이 드러난다. 생물무기와 전염병의 공포를 현실감 있게 묘사한 작품이다.",
            "author": "Michael Crichton",
            "year": 1969
        },
        {
            "name": "The Left Hand of Darkness",
            "description": "외교관 제너리 아이가 젠더가 유동적인 외계 종족이 사는 행성에 파견된다. 그는 그들과의 문화적 충돌과 이해를 통해 진정한 인간성과 연대를 배워간다. 젠더와 정체성, 타자성에 대한 철학적 탐구가 돋보이는 작품이다.",
            "author": "Ursula K. Le Guin",
            "year": 1969
        },
        {
            "name": "The Three-Body Problem",
            "description": "문화대혁명기의 중국에서 외계 문명과의 접촉이 시작되며 인류는 큰 위기에 직면한다. 세 개의 태양이 있는 불안정한 행성에 사는 외계인과의 접촉이 인류 문명에 위협을 가한다. 과학, 철학, 문명 충돌을 고찰하는 하드 SF 걸작이다.",
            "author": "Liu Cixin",
            "year": 2008
        }
    ]
    # page_content: embedding vector로 변환할 text를 저장.
    # metadata: 이 문서에 관련 정보들을 추가.
    documents = [
        Document(page_content=doc['description'], metadata={key:value for key, value in doc.items() if key != "description"}, id=str(uuid4())) for doc in data
    ]
    return documents


documents = get_documents()
documents

[Document(id='ffd9a10e-2aa5-48bb-8c8d-58a7d4c7be7e', metadata={'name': 'The Time Machine', 'author': 'H.G. Wells', 'year': 1895}, page_content='시간 여행자는 미래로 여행을 떠나 인류의 진화와 몰락을 목격하게 된다. 그는 멀리 떨어진 미래 사회에서 인간 문명이 퇴화한 모습을 보고 충격을 받는다. 이 이야기는 인간 문명과 과학기술의 진보가 가져올 결과를 철학적으로 탐구한다.'),
 Document(id='78a482a2-5d98-4ca7-8822-3ac6aa798463', metadata={'name': "Ender's Game", 'author': 'Orson Scott Card', 'year': 1985}, page_content="소년 엔더는 외계 종족 '버거'와의 전쟁을 대비해 혹독한 군사 훈련을 받는다. 그는 전략적 천재성을 발휘하여 점차 전쟁의 중심에 서게 된다. 이야기는 어린아이를 도구로 삼는 전쟁의 비인간성과 심리적 고통을 그린다."),
 Document(id='e340a75c-0ac3-4823-a1d8-846ad4d5c9ae', metadata={'name': 'Brave New World', 'author': 'Aldous Huxley', 'year': 1932}, page_content='유전자 조작과 세뇌로 계급이 철저히 나뉘어진 미래 사회가 배경이다. 인간은 자유 없이 쾌락과 안정을 주입받으며 살고, 비순응자는 배제된다. 디스토피아 속에서 인간 본성과 자유의 의미를 묻는 소설이다.'),
 Document(id='bddcc0bb-4e1a-4c83-8e67-7b163375cecf', metadata={'name': "The Hitchhiker's Guide to the Galaxy", 'author': 'Douglas Adams', 'year': 1979}, page_content='지구 멸망 직후 우주로 탈출한 주인공이 외계 친구

In [2]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# QdrantDB와 연결, Collection 생성
COLLECTION_NAME = "SF_BOOKS"
VECTOR_SIZE = 3072  # openai text-embedding-3-large의 임베딩벡터 차원수

# 연결 - qdrant db 실행
client = QdrantClient(url="http://localhost:6333")

# collection이 있으면 삭제하고 새로 생성.
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

# 생성
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)
client.close()

In [4]:
################################################################################
# Vector Store 생성
#  - Langchain에서 Vector Database와 연동하는 모듈 -> VectorDB마다 다른 클래스를 사용. -> 같은방식으로 사용
################################################################################

# Qdrant 와 연결
client = QdrantClient(url="http://localhost:6333")

# Embedding 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# VectorStore 생성 - db와 연동
vectorstore = QdrantVectorStore(
    client=client, # 연동할 qdrant client
    embedding=embeddings, # 임베딩 모델. (upsert/search 할때 embedding 처리할 모델.)
    collection_name=COLLECTION_NAME
)

##############################
#  데이터셋 upsert
##############################
## vectorstore.add_document(list[Document]) Document.page_content: Vector
ids = vectorstore.add_documents(documents) # upsert한 데이터들의 id들을 반환
print(ids)

['ffd9a10e-2aa5-48bb-8c8d-58a7d4c7be7e', '78a482a2-5d98-4ca7-8822-3ac6aa798463', 'e340a75c-0ac3-4823-a1d8-846ad4d5c9ae', 'bddcc0bb-4e1a-4c83-8e67-7b163375cecf', 'f21eebbd-63b8-4304-b539-c4d2d13f2e9a', 'd96bca67-b955-4632-9346-90f607da86dd', 'c6e82c1c-b3c5-4ae9-aad7-ff97a136bc9b', '5c3d6b0f-3085-40a3-9780-e47522ab60d6', 'f6a29542-045f-4022-bf2a-0b6aa10d4f4d', '5db3bb03-5755-4709-9e94-058bba19ec8a', 'ee20e30f-dd47-4665-8376-8de7f9ba1020', '8d1a2daf-d72b-410e-bf80-1001ce3bcd32', '2e06ea46-1b1c-4ed2-99b0-5f66896d6ec9', 'c48a3f5d-6761-4b7d-8842-19d339311c06', 'ebd29900-8567-4449-8367-13d14d4aa287', '77cf49bd-af7e-4058-9b7f-f10d9edbfed6', 'e2052ca7-e878-4f12-976f-e7608b566b9c', 'd33c8ca2-3d52-4364-9b3c-b15cc984ff00', 'dc8388c7-3430-4349-a5f7-b2ea8e1a13f2', 'b6ebbcc9-e9a6-4f2d-a434-8d2e98ceea84']


In [5]:
#############################
# Vector Store(Collection)에 대한 정보 조회
#############################
print(vectorstore.collection_name)
print(vectorstore.distance)
# page_content, metadata가 모두 payload로 저장된다.
print(vectorstore.content_payload_key)  # page_content의 payload key값
print(vectorstore.metadata_payload_key) # metadata의 payload key값

SF_BOOKS
Cosine
page_content
metadata


In [6]:
# Qdrant Client를 VectorStore에서 조회 -> Qdrant API를 직접 사용할 경우.
client2 = vectorstore.client
client2.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='SF_BOOKS'), CollectionDescription(name='SF_BOOKS_HYBRID'), CollectionDescription(name='SF_BOOK_SPARSE'), CollectionDescription(name='test_collection2')])

In [7]:
# Payload Index 생성
from qdrant_client.models import PayloadSchemaType

client2.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name='metadata.year',
    field_schema=PayloadSchemaType.INTEGER    
)

client2.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name='metadata.name',
    field_schema=PayloadSchemaType.TEXT    
)

client2.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name='metadata.author',
    field_schema=PayloadSchemaType.KEYWORD    
)

UpdateResult(operation_id=7, status=<UpdateStatus.COMPLETED: 'completed'>)

# 검색

In [8]:
# vector store를 이용해서 의미기반(유사도) 검색
query = "암울한 미래사회를 그린 소설을 추천해줘."
result = vectorstore.similarity_search(
    query=query,
    k=5
    , score_threshold=0.45  # score가 0.45 이상인 것만 반환
)

In [10]:
result = vectorstore.similarity_search_with_score(
    query=query,
    k=5,
    score_threshold=0.45   # score가 0.45 이상인 것만 반환
)

In [11]:
# list[Document]
# tuple[Document, float:점수]
result

[(Document(metadata={'name': 'Brave New World', 'author': 'Aldous Huxley', 'year': 1932, '_id': 'e340a75c-0ac3-4823-a1d8-846ad4d5c9ae', '_collection_name': 'SF_BOOKS'}, page_content='유전자 조작과 세뇌로 계급이 철저히 나뉘어진 미래 사회가 배경이다. 인간은 자유 없이 쾌락과 안정을 주입받으며 살고, 비순응자는 배제된다. 디스토피아 속에서 인간 본성과 자유의 의미를 묻는 소설이다.'),
  0.4833274),
 (Document(metadata={'name': 'The Time Machine', 'author': 'H.G. Wells', 'year': 1895, '_id': 'ffd9a10e-2aa5-48bb-8c8d-58a7d4c7be7e', '_collection_name': 'SF_BOOKS'}, page_content='시간 여행자는 미래로 여행을 떠나 인류의 진화와 몰락을 목격하게 된다. 그는 멀리 떨어진 미래 사회에서 인간 문명이 퇴화한 모습을 보고 충격을 받는다. 이 이야기는 인간 문명과 과학기술의 진보가 가져올 결과를 철학적으로 탐구한다.'),
  0.47456366)]

In [15]:
################################################
# Payload Filtering
#  - 조건 정의: Qdrant Filter 객체를 이용.
################################################
from qdrant_client.models import Filter, FieldCondition, Range
filter_condition = Filter(
    must=[
        FieldCondition(key="metadata.year", range=Range(lte=1900)) # metadata.year <= 1900
    ]
)
query = "암울한 미래사회를 그린 소설을 추천해줘."
result = vectorstore.similarity_search_with_score(
    query = query,
    k=5,
    filter=filter_condition
)
result

[(Document(metadata={'name': 'The Time Machine', 'author': 'H.G. Wells', 'year': 1895, '_id': 'ffd9a10e-2aa5-48bb-8c8d-58a7d4c7be7e', '_collection_name': 'SF_BOOKS'}, page_content='시간 여행자는 미래로 여행을 떠나 인류의 진화와 몰락을 목격하게 된다. 그는 멀리 떨어진 미래 사회에서 인간 문명이 퇴화한 모습을 보고 충격을 받는다. 이 이야기는 인간 문명과 과학기술의 진보가 가져올 결과를 철학적으로 탐구한다.'),
  0.47456366),
 (Document(metadata={'name': 'The Island of Doctor Moreau', 'author': 'H.G. Wells', 'year': 1896, '_id': '77cf49bd-af7e-4058-9b7f-f10d9edbfed6', '_collection_name': 'SF_BOOKS'}, page_content='외딴 섬에서 미친 과학자 모로 박사는 동물들을 인간처럼 개조하는 실험을 진행한다. 주인공은 이 섬에서 인간과 짐승의 경계가 무너진 광경을 목격한다. 생명 윤리와 과학의 오만함을 날카롭게 비판하는 소설이다.'),
  0.41777277),
 (Document(metadata={'name': 'The War of the Worlds', 'author': 'H.G. Wells', 'year': 1898, '_id': 'c48a3f5d-6761-4b7d-8842-19d339311c06', '_collection_name': 'SF_BOOKS'}, page_content='화성인이 지구를 침략하면서 인류는 공포와 절망 속에 빠진다. 압도적인 기술력 앞에 속수무책으로 무너지는 인간의 모습이 그려진다. 외계 생명체와 문명의 충돌을 통해 인간 중심적 세계관에 경종을 울린다.'),
  0.3900504),
 (Document(metadata={'name': 'Th

In [16]:
client.close()

# Qdrant 세가지 방식

- **Dense Vector Search** (default): **의미 기반** 검색. Vector간 의미적 유사도록 검색
- **Sparse Vector Search**: **키워드 기반** 검색.
- **Hybrid Search**: 두가지 방법을 합쳐서 검색

## Dense Vector Search
- Default 방식
- `RetrievalMode.DENSE` 로 설정
- embedding 파라미터에 값이 제공되야 한다.

In [36]:
!uv pip install fastembed

Resolved 37 packages in 350ms
 Downloaded onnxruntime
Prepared 7 packages in 689ms
Installed 7 packages in 509ms
 + fastembed==0.8.0
 + flatbuffers==25.12.19
 + loguru==0.7.3
 + mmh3==5.2.1
 + onnxruntime==1.27.0
 + py-rust-stemmers==0.1.8
 + win32-setctime==1.2.0


### Sparse Vector Search
- **희소 표현(sparse representation)을** 사용한 벡터 검색 방식.
- 이 기능은 특히 BM25, TF-IDF, Bag-of-Words 검색과 같은 전통적인 정보 검색(IR: Information Retrieval) 기법을 이용해 검색을 한다.
- 보통 이 방식을 단독으로 쓰지 않고 의미기반 검색을 보완하기 위해 사용한다. (Hybrid 검색)
- `pip install -qU fastembed`
  
#### 설정
- sparse vector search 만 할 경우
  - **collection 생성** 시 `sparse_vectors_config` 에 sparse vector 설정을 한다.
  - Langchain VectorStore 생성시 `sparse_embedding_model`을 설정하고 collection 생성시 지정한 설정을 `sparse_vector_name` 파라미터를 이용해 전달한다.

#### BM25 (Best Matching 25) 알고리즘

BM25는 검색 엔진이나 추천 시스템에서 **사용자가 입력한 검색어와 가장 관련성이 높은 문서**를 찾아 순위를 매기는 데 사용되는 알고리즘이다.   
통계 기반의 정보 검색 기법 중 하나로, 특히 문서 간의 관련성을 정량적으로 평가할 때 많이 사용된다.

BM25는 다음 세 가지 주요 요소를 바탕으로 점수를 계산한다:


##### 1. **단어의 출현 빈도 (Term Frequency, TF)**

- 문서 내 특정 단어가 얼마나 자주 등장하는지를 측정한다.
- 사용자가 검색한 단어가 어떤 문서에서 많이 등장할수록, 해당 문서는 검색어와 관련성이 높다고 판단한다.
- 하지만 너무 자주 등장하는 단어에 대해서는 **점수 증가를 둔화**시키는 방식(로그나 분수 기반의 함수)을 사용한다. 즉, 단어의 빈도 수가 많다고 무한정 점수가 올라가지는 않는다.

##### 2. **단어의 희소성 (Inverse Document Frequency, IDF)**

- 전체 문서(검색 대상 문서) 집합에서 특정 단어가 얼마나 희귀한지를 평가한다.
- 많은 문서에 동시에 등장하는 단어는 정보성이 낮기 때문에 낮은 점수를 부여하고, 드물게 등장하는 단어에는 높은 점수를 부여한다.

##### 3. **문서의 길이 보정 (Document Length Normalization)**

- 문서의 길이에 따라 점수를 보정한다.
- 같은 단어가 등장한 횟수가 같더라도, 짧은 문서에서는 그 단어의 중요도가 더 높다고 판단한다.
- 너무 긴 문서가 단지 길이 때문에 더 높은 점수를 받는 것을 방지하기 위해, 평균 문서 길이를 기준으로 보정한다.

### BM25 점수 계산 과정

1. **검색어 토큰화**
   - 사용자가 입력한 검색어 문장을 단어(토큰) 단위로 분리한다.
   - 예: "강아지 산책 방법" → \["강아지", "산책", "방법"\]
2. **각 문서에 대해 단어별 점수 계산**
   - 분리된 각 단어에 대해, 위 세 가지 요소를 기반으로 각 대상 문서의 점수를 계산한다:
3. **문서별 총점 산출**
   - 각 단어별 점수를 모두 더해 해당 문서의 총점을 계산한다.
4. **문서 순위화**
   - 점수가 높은 문서일수록 검색어와 관련성이 높다고 판단하여, 그 순서대로 결과를 정렬한다.



### BM25의 한계

- **의미 파악 불가**: BM25는 단어의 단순 일치 여부만 평가하며, 단어의 의미나 문맥은 고려하지 않는다. 예를 들어 "사과"가 "과일"인지 "용서"인지 구분하지 못한다.
- **동의어 처리 어려움**: "자동차"와 "승용차"처럼 다른 단어지만 유사한 의미를 가진 경우, BM25는 이들을 동일하게 인식하지 못한다.
- **자연어 이해 부족**: 문장의 구조나 어순, 구문 정보는 활용하지 않는다.
- **언어별 특성**- 한국어처럼 교착어에서는 형태소 분석이 중요한데 이에 대한 고려가 부족
- **구문 관계**: 단어 간의 거리나 위치 정보를 고려하지 않음

BM25는 간단하면서도 강력한 정보 검색 알고리즘으로, **키워드 기반 검색**에 매우 효과적이다. 다만 단어의 **의미나 문맥을 이해하지 못한다는 점**에서 **의미 기반 검색 모델**이 보완적으로 사용되기도 한다.


In [37]:
from langchain_qdrant import FastEmbedSparse, RetrievalMode, QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

import os
from huggingface_hub import login

load_dotenv()
login(os.getenv("HUGGINGFACE_API_KEY")) # 허깅페이스 로그인

In [39]:
COLLECTION_NAME = "SF_BOOK_SPARSE"
sparse_embeddings = FastEmbedSparse(model_name="qdrant/bm25")

client = QdrantClient(url="http://localhost:6333")

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

# 컬렉션 생성 - Sparse Vector 저장소만 가지는 collection을 생성.
client.create_collection(
    collection_name=COLLECTION_NAME,
    # Sparse Vector 저장소 설정.
    sparse_vectors_config = { # key(저장소의 이름)
        "sparse":SparseVectorParams(index=SparseIndexParams(on_disk=False)) 
         # Index로 쓰이는 sparse vector (index)를 메모리(False)에 저장할지 디스크(True)에 저장할 지 설정
    }
)


True

In [40]:
# VectorStore
sparse_vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME, # Spare Vector Index
    sparse_embedding=sparse_embeddings,
    sparse_vector_name="sparse", #Spare Vector 저장소의 이름->sparse_vector_config에 설정한 이름
    retrieval_mode=RetrievalMode.SPARSE
)

# 데이터 upsert
sparse_vectorstore.add_documents(documents=documents)

['5c056838-3775-489a-a2fe-66d802126589',
 '25d00aad-f14e-4ad7-bdba-942291ed1d3a',
 '9f8383c5-f5ff-4f64-aac1-8199d64ee1a2',
 '659c977f-f5d2-4edc-af73-1ca76a5578d4',
 '44e88c19-cb5c-4011-b355-2dbc1f40db0c',
 '514e8249-2898-45a7-ad34-b439abe95657',
 '24cd93bc-0f5d-4339-83a1-eaca0f888155',
 'da6c3afd-3733-419f-8f27-e33768b626e4',
 'bf9086bd-6ef7-4724-a8fd-2cfd4fcbc06e',
 '3e2269c3-147e-4751-af6e-67eeeb65a832',
 'ebb8cea0-4e52-4a20-97e4-7ecc0deaed3f',
 '6b534b59-bd1d-4fdc-80f2-0f7797eb8fc8',
 '0d87b475-d021-4630-ac7e-24fd8f053d6e',
 '015530ef-55ee-489d-9359-4720449eae83',
 '2fce2740-0cc1-48a7-9201-89f1d67b1c33',
 '617dea59-93bb-4e5a-a9b5-1440b2e09e01',
 '99ce0925-c387-4aed-a546-82f8de603fde',
 'b37e306f-a719-41a2-8568-0f5a68dd0cee',
 '40ef910a-4b5a-4f9d-a6f6-7a9c59954bd8',
 '915a36c4-7a40-499f-a37c-cbaf5ec277e8']

In [43]:
query = "암울한 미래세계에 대한 소설을 추천해줘."
query = "우주 여행과 외계 생명체에 대한 소설을 소개해줘."
result = sparse_vectorstore.similarity_search_with_score(
    query=query,
    k=5
)

In [44]:
result

[(Document(metadata={'name': "The Hitchhiker's Guide to the Galaxy", 'author': 'Douglas Adams', 'year': 1979, '_id': '659c977f-f5d2-4edc-af73-1ca76a5578d4', '_collection_name': 'SF_BOOK_SPARSE'}, page_content='지구 멸망 직후 우주로 탈출한 주인공이 외계 친구와 함께 기상천외한 모험을 시작한다. 우주의 황당한 존재들과 철학적 농담들이 넘치는 코믹 SF다. 우주에 대한 진지한 질문을 유쾌하게 풀어낸 작품이다.'),
  3.1542985),
 (Document(metadata={'name': 'The Left Hand of Darkness', 'author': 'Ursula K. Le Guin', 'year': 1969, '_id': '40ef910a-4b5a-4f9d-a6f6-7a9c59954bd8', '_collection_name': 'SF_BOOK_SPARSE'}, page_content='외교관 제너리 아이가 젠더가 유동적인 외계 종족이 사는 행성에 파견된다. 그는 그들과의 문화적 충돌과 이해를 통해 진정한 인간성과 연대를 배워간다. 젠더와 정체성, 타자성에 대한 철학적 탐구가 돋보이는 작품이다.'),
  3.1463687),
 (Document(metadata={'name': 'The War of the Worlds', 'author': 'H.G. Wells', 'year': 1898, '_id': '015530ef-55ee-489d-9359-4720449eae83', '_collection_name': 'SF_BOOK_SPARSE'}, page_content='화성인이 지구를 침략하면서 인류는 공포와 절망 속에 빠진다. 압도적인 기술력 앞에 속수무책으로 무너지는 인간의 모습이 그려진다. 외계 생명체와 문명의 충돌을 통해 인간 중심적 세계관에 경종을 울린다.'),
  1.5811342),
 

In [45]:
client.close()

### Hybrid
- Hybrid Search는 Dense Vector Search와 Sparse Vector Search를 결합하여, **의미 기반 검색(semantic search)**과 **정확한 키워드 검색(lexical search)**의 장점을 모두 활용하는 검색 방식이다.
- RAG(Retrieval-Augmented Generation) 시스템이나 검색 기반 질문응답 시스템에서 정확도와 재현율을 동시에 향상시키는 데 매우 유용하다.

> - Dense Vector: 문장의 의미를 파악하는 임베딩 기반 검색
> - Sparse Vector: 키워드 기반의 전통 정보 검색(BM25, TF-IDF 등)

- Dense vector만 사용하면 문맥적 의미는 이해하지만 키워드가 누락된 문서가 나올 수있다.
- Sparse vector만 사용하면 정확한 단어, 키워드는 찾지만 의미 파악이 부족할 수있다.
- 위 두가지 방식의 단점을 보완한 것이 Hybrid 방식으로 **정확한 키워드와 의미적으로 유사한 문서**를 함께 찾는다.
- 하이브리드 방식을 적용하면
  - BM25로 1차 후보 검색
  - Dense Embedding으로 의미 기반 재정렬

In [47]:
COLLECTION_NAME = "SF_BOOKS_HYBRID"
# 임베딩 모델 - SPARSE/DENSE
sparse_embeddings = FastEmbedSparse(model_name="qdrant/bm25")
dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
VECTOR_SIZE = 3072

client = QdrantClient(url="http://localhost:6333")

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

# Collection 생성 -> INDEX는 DENSE, SPARSE 둘 모두 사용.
client.create_collection(
    collection_name=COLLECTION_NAME,
    # sparse vector 저장소 설정
    sparse_vectors_config={
        "sparse":SparseVectorParams(index=SparseIndexParams(on_disk=False)),
    },
    # dense vector 저장소 설정
    vectors_config={
        "dense":VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    }
)

True

In [49]:
hybrid_vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,

    # embedding 모델 설정 - Dense/Sparse
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,

    # Vector저장소 이름
    vector_name="dense", # Dense Vector저장소이름
    sparse_vector_name="sparse",

    # 조회방식
    retrieval_mode=RetrievalMode.HYBRID
)

# 데이터저장
hybrid_vectorstore.add_documents(documents)

['5c056838-3775-489a-a2fe-66d802126589',
 '25d00aad-f14e-4ad7-bdba-942291ed1d3a',
 '9f8383c5-f5ff-4f64-aac1-8199d64ee1a2',
 '659c977f-f5d2-4edc-af73-1ca76a5578d4',
 '44e88c19-cb5c-4011-b355-2dbc1f40db0c',
 '514e8249-2898-45a7-ad34-b439abe95657',
 '24cd93bc-0f5d-4339-83a1-eaca0f888155',
 'da6c3afd-3733-419f-8f27-e33768b626e4',
 'bf9086bd-6ef7-4724-a8fd-2cfd4fcbc06e',
 '3e2269c3-147e-4751-af6e-67eeeb65a832',
 'ebb8cea0-4e52-4a20-97e4-7ecc0deaed3f',
 '6b534b59-bd1d-4fdc-80f2-0f7797eb8fc8',
 '0d87b475-d021-4630-ac7e-24fd8f053d6e',
 '015530ef-55ee-489d-9359-4720449eae83',
 '2fce2740-0cc1-48a7-9201-89f1d67b1c33',
 '617dea59-93bb-4e5a-a9b5-1440b2e09e01',
 '99ce0925-c387-4aed-a546-82f8de603fde',
 'b37e306f-a719-41a2-8568-0f5a68dd0cee',
 '40ef910a-4b5a-4f9d-a6f6-7a9c59954bd8',
 '915a36c4-7a40-499f-a37c-cbaf5ec277e8']

In [54]:
query = "암울한 미래세계에 대한 소설을 추천해줘."
query = "우주 여행과 외계 생명체에 대한 소설을 소개해줘."

result = hybrid_vectorstore.similarity_search_with_score(
    query=query,
    k=5
)
print(result)
client.close()

ResponseHandlingException: Cannot send a request, as the client has been closed.